In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder, TargetEncoder
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, roc_curve

In [2]:
df = pd.read_csv("customer_booking.csv", encoding="ISO-8859-1")

In [3]:
df.head()

,num_passengers,sales_channel,trip_type,purchase_lead,length_of_stay,flight_hour,flight_day,route,booking_origin,wants_extra_baggage,wants_preferred_seat,wants_in_flight_meals,flight_duration,booking_complete
0,2,Internet,RoundTrip,262,19,7,Sat,AKLDEL,New Zealand,1,0,0,5.52,0
1,1,Internet,RoundTrip,112,20,3,Sat,AKLDEL,New Zealand,0,0,0,5.52,0
2,2,Internet,RoundTrip,243,22,17,Wed,AKLDEL,India,1,1,0,5.52,0
3,1,Internet,RoundTrip,96,31,4,Sat,AKLDEL,New Zealand,0,0,1,5.52,0
4,2,Internet,RoundTrip,68,22,15,Wed,AKLDEL,India,1,0,1,5.52,0


In [4]:
df.isnull().sum()

num_passengers           0
sales_channel            0
trip_type                0
purchase_lead            0
length_of_stay           0
flight_hour              0
flight_day               0
route                    0
booking_origin           0
wants_extra_baggage      0
wants_preferred_seat     0
wants_in_flight_meals    0
flight_duration          0
booking_complete         0
dtype: int64

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   num_passengers         50000 non-null  int64  
 1   sales_channel          50000 non-null  object 
 2   trip_type              50000 non-null  object 
 3   purchase_lead          50000 non-null  int64  
 4   length_of_stay         50000 non-null  int64  
 5   flight_hour            50000 non-null  int64  
 6   flight_day             50000 non-null  object 
 7   route                  50000 non-null  object 
 8   booking_origin         50000 non-null  object 
 9   wants_extra_baggage    50000 non-null  int64  
 10  wants_preferred_seat   50000 non-null  int64  
 11  wants_in_flight_meals  50000 non-null  int64  
 12  flight_duration        50000 non-null  float64
 13  booking_complete       50000 non-null  int64  
dtypes: float64(1), int64(8), object(5)
memory usage: 5.3+ 

In [6]:
df.shape

(50000, 14)

In [7]:
day_mapping = {
    "Mon": 1, "Tue": 2, "Wed": 3, "Thu": 4, "Fri": 5, "Sat": 6, "Sun": 7
}
df["flight_day"] = df["flight_day"].map(day_mapping)

In [8]:
df['wants_extra_services'] = (
    df['wants_extra_baggage'] + 
    df['wants_preferred_seat'] + 
    df['wants_in_flight_meals']
)

df['is_weekend'] = df['flight_day'].isin([6, 7]).astype(int)

In [9]:
X = df.drop(columns=['booking_complete'])
y = df['booking_complete']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y, shuffle=True
)

In [10]:
low_card_cols = ['sales_channel', 'trip_type']
ord_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_train[low_card_cols] = ord_encoder.fit_transform(X_train[low_card_cols])
X_test[low_card_cols] = ord_encoder.transform(X_test[low_card_cols])

In [11]:
target_enc = TargetEncoder(smooth="auto", random_state=42)

X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

high_card_cols = ['route', 'booking_origin']
X_train_encoded[high_card_cols] = target_enc.fit_transform(X_train[high_card_cols], y_train)
X_test_encoded[high_card_cols] = target_enc.transform(X_test[high_card_cols])

In [12]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [13]:
param_grid = {
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'n_estimators': [100, 200]
}

In [14]:
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid=param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

In [15]:
grid_search.fit(X_train_encoded, y_train)
best_rf = grid_search.best_estimator_
print(f"Best Parameters: {grid_search.best_params_}\n")

Best Parameters: {'max_depth': 10, 'min_samples_split': 10, 'n_estimators': 100}



In [16]:
y_pred = best_rf.predict(X_test_encoded)
y_prob = best_rf.predict_proba(X_test_encoded)[:, 1]

print("=================== TEST SET EVALUATION ===================")
print(f"Test ROC AUC Score: {roc_auc_score(y_test, y_prob):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))

=================== TEST SET EVALUATION ===================
Test ROC AUC Score: 0.7963

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.72      0.82      8504
           1       0.31      0.73      0.44      1496

    accuracy                           0.72     10000
   macro avg       0.63      0.72      0.63     10000
weighted avg       0.84      0.72      0.76     10000



In [17]:
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc_score(y_test, y_prob):.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.savefig('roc_curve.png', dpi=300)
plt.close()

In [18]:
importances = best_rf.feature_importances_
indices = np.argsort(importances)[::-1][:10] # Top 10

plt.figure(figsize=(10, 5))
sns.barplot(x=importances[indices], y=X.columns[indices], palette="viridis")
plt.title("Top 10 Drivers of Customer Bookings")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300)
plt.close()

print("\nProcessing Complete! Charts 'roc_curve.png' and 'feature_importance.png' saved successfully.")

C:\Users\Shankhni Marandi\AppData\Local\Temp\ipykernel_26576\2923176227.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=importances[indices], y=X.columns[indices], palette="viridis")



Processing Complete! Charts 'roc_curve.png' and 'feature_importance.png' saved successfully.
